# Ngày 10 · KNN và lời nguyền số chiều

- Cài KNN vector hóa bằng khoảng cách cặp của ngày 1
- Thấy vì sao KNN bắt buộc chuẩn hóa thang đo
- Đo độ tương phản khoảng cách gần nhất/xa nhất khi số chiều tăng

**Đọc:**
- ISLR, §2.2.3 The Classification Setting (đoạn K-Nearest Neighbors) và §3.5
- Vũ Hữu Tiệp, Machine Learning cơ bản, Chương 9 K lân cận

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))
# Breast Cancer Wisconsin (Diagnostic), UCI, có sẵn trong scikit-learn. 569 mẫu, 30 đặc trưng.
_bc = load_breast_cancer()
X_bc = _bc.data
y_bc = 1 - _bc.target   # 1 = ác tính (lớp cần phát hiện), 0 = lành tính
feature_names = list(_bc.feature_names)

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))

# Một phép chia dùng chung cho cả tuần: 70% train, 30% validation, phân tầng theo nhãn.
idx_tr, idx_va = train_test_split(np.arange(len(y_bc)), test_size=0.3, random_state=0, stratify=y_bc)
_scaler = StandardScaler().fit(X_bc[idx_tr])
X_tr, X_va = _scaler.transform(X_bc[idx_tr]), _scaler.transform(X_bc[idx_va])
y_tr, y_va = y_bc[idx_tr], y_bc[idx_va]
from sklearn.neighbors import KNeighborsClassifier
def _knn_ok(fn, k):
    ref = KNeighborsClassifier(n_neighbors=k, algorithm='brute').fit(X_tr, y_tr).predict(X_va)
    return np.array_equal(np.asarray(fn(X_tr, y_tr, X_va, k)), ref)


## Dự đoán KNN

Gợi ý: Tính ma trận khoảng cách (n_test × n_train) bằng broadcasting, lấy k cột nhỏ nhất mỗi hàng, đếm phiếu bằng np.bincount.

In [ ]:
def knn_predict(X_train, y_train, X_test, k):
    # Nhãn là số nguyên 0..C-1. Hòa phiếu thì chọn nhãn nhỏ hơn.
    raise NotImplementedError


In [ ]:
_report('d10-knn',
    _check('Khớp scikit-learn với k = 1', lambda: (_knn_ok(knn_predict, 1), 'Trên dữ liệu đã chuẩn hóa.')),
    _check('Khớp scikit-learn với k = 5 và k = 15', lambda: (_knn_ok(knn_predict, 5) and _knn_ok(knn_predict, 15), 'Bỏ phiếu đa số.')),
)


## Đo lời nguyền số chiều

Gợi ý: Sinh n điểm đều trong [0,1]^d bằng rng.random((n, d)). Với mỗi điểm truy vấn, tính (d_max − d_min)/d_min tới các điểm còn lại.

In [ ]:
def distance_contrast(n, d, seed, queries=20):
    # Trung bình (d_max - d_min) / d_min trên `queries` điểm truy vấn đầu tiên.
    raise NotImplementedError


In [ ]:
_report('d10-curse',
    _check('Độ tương phản giảm mạnh khi d tăng', lambda: ((lambda a, b, c: a > b > c > 0 and a > 5 * c)(distance_contrast(200, 2, 0), distance_contrast(200, 50, 0), distance_contrast(200, 1000, 0)), 'd = 2, 50, 1000.')),
    _check('Cùng seed cho cùng kết quả', lambda: (distance_contrast(100, 10, 3) == distance_contrast(100, 10, 3), 'Dùng np.random.default_rng(seed).')),
)


## Dự án

Trên Titanic, chọn hai cột số (Age, Fare). Thử KNN với và không chuẩn hóa bằng cross_val_score; ghi accuracy và giải thích vì sao Fare lấn át Age khi chưa chuẩn hóa.